In [1]:
import networkx as nx
import random
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import gzip
import csv
from pies import pies_sampling
from ties import ties_sampling
from scipy.stats import ks_2samp
random.seed(42)
np.random.seed(42)
# url https://www.researchgate.net/publication/254639513_Network_Sampling_via_Edge-based_Node_Selection_with_Graph_Induction

In [2]:
# demo data https://snap.stanford.edu/data/soc-sign-bitcoin-otc.html
filename = 'data/soc-sign-bitcoinotc.csv.gz'
data = []
with gzip.open(filename, 'rt') as f:
    for row in csv.reader(f):
        data.append(row)
data = np.array(data)
df = pd.DataFrame(data,columns=['source','target','rating','time'])
G = nx.DiGraph()
G.add_edges_from(df[['source','target']].values)
# add time and rating to the edges
for i in range(len(df)):
    G[df['source'][i]][df['target'][i]]['time'] = df['time'][i]
    G[df['source'][i]][df['target'][i]]['rating'] = df['rating'][i]

In [3]:
class Benchmark:

    def T1(self):
        pass

    def T2(self):
        pass

    def T3(self):
        pass

    def T4(self):
        pass

    def T5(self):   
        pass

In [4]:
class Helper:


    def __init__(self):
        pass


    @staticmethod   
    def make_snapshots_nodes(G, N):
        """
        Creates snapshots of the graph based on the number of nodes.

        Args:
            G (nx.Graph or nx.DiGraph): The input graph.
            N (int): The number of snapshots to create.

        Returns:
            list of nx.Graph or nx.DiGraph: A list of graph snapshots.
        """
        # sort G on Time attribute
        G = nx.DiGraph(sorted(G.edges(data=True), key=lambda x: x[2]['time']))
        nodes = list(G.nodes)
        chunk_size = len(nodes) // N  # Base chunk size
        leftover = len(nodes) % N  # Number of extra nodes to distribute

        # print(f'Base chunk size {chunk_size}, Leftover {leftover}')
        
        # Split the graph into N snapshots
        snapshots = []
        snapshot_nodes = []
        for i in range(N):
            snapshot_nodes_local = nodes[i * chunk_size: (i + 1) * chunk_size]
            snapshot_nodes = snapshot_nodes_local + snapshot_nodes
            if i < leftover:
                snapshot_nodes.append(nodes[N * chunk_size + i])
            snapshot_graph = G.subgraph(snapshot_nodes)
            snapshots.append(snapshot_graph.copy())
        return snapshots
    
    @staticmethod
    def make_snapshots_edges(G, N):
        edges = list(G.edges(data=True))
        chunk_size = len(edges) // N
        print(f'chunk size {chunk_size}')
        edge_chunks = [edges[i * chunk_size: (i + 1) * chunk_size] for i in range(N)]
        leftover = len(edges) % N
        for i in range(leftover):
            edge_chunks[i].append(edges[N * chunk_size + i])
        snapshots = []
        snapshot_graph = nx.DiGraph()
        for chunk in edge_chunks:
            snapshot_graph.add_edges_from(chunk)
            snapshots.append(snapshot_graph.copy())
        return snapshots

In [5]:
Gs = pies_sampling(G,0.1)
Gt = Helper.make_snapshots_nodes(G,5)
fractions = [len(snapshot.nodes()) / len(G.nodes()) for snapshot in Gt]
St = [ties_sampling(G,fraction) for fraction in fractions]
print([len(snapshot.nodes())/len(G.nodes) for snapshot in St])
print([len(snapshot.nodes())/len(G.nodes) for snapshot in Gt])
print([len(snapshot.edges()) for snapshot in St])
print([len(snapshot.edges()) for snapshot in Gt])
fractions

[0.20013603128719606, 0.400102023465397, 0.6002380547525931, 0.800034007821799, 1.0]
[0.20013603128719606, 0.400102023465397, 0.600068015643598, 0.800034007821799, 1.0]
[14383, 24122, 29824, 33396, 35592]
[6456, 15235, 21968, 30794, 35592]


[0.20013603128719606,
 0.400102023465397,
 0.600068015643598,
 0.800034007821799,
 1.0]

In [6]:
# new benchmark ideas temproal
# Edge Lifetimes: Check the overlap in edges present across time.
# Node Activity: Compare the temporal activity patterns of nodes., see how nodes expand their list of neighbors
# Temporal Motifs: Analyze the frequency of temporal motifs (e.g., triadic closure over time). (ik dacht dat dit heel traag is)
# Entropy of Node Degrees: Compute entropy for degree distributions in both sets of graphs
# Wasserstein Distance?

# new benchmark ideas directed
# Edge Density Over Time: Compare the edge density of the graph over time.
# Pagerank
# HITS: identify hubs and authorities in the graph
# Flow Efficiency Measure the efficiency of information or resource transfer along directed paths:

In [7]:
import networkx as nx

def b_flow_efficiency(G: nx.DiGraph) -> float:
    """
    Calculate the Flow Efficiency of a directed graph.
    
    Flow Efficiency is defined as the sum of the reciprocal shortest path lengths 
    between all pairs of nodes (i, j) in the graph, divided by the total possible 
    number of node pairs.
    
    Parameters:
        G (nx.DiGraph): A NetworkX directed graph.
    
    Returns:
        float: The flow efficiency of the graph.
    """
    # Number of nodes in the graph
    n = G.number_of_nodes()
    
    # Total possible pairs of nodes
    total_pairs = n * (n - 1)
    
    # Handle the case of an empty graph or a single node
    if total_pairs == 0:
        return 0.0
    
    # Calculate reciprocal of shortest path lengths
    efficiency_sum = 0
    for source in G.nodes():
        # Get shortest path lengths from the source node
        shortest_paths = nx.single_source_shortest_path_length(G, source)
        for target, length in shortest_paths.items():
            if source != target and length > 0:
                efficiency_sum += 1 / length
    
    # Compute flow efficiency
    flow_efficiency = efficiency_sum / total_pairs
    return flow_efficiency

# Calculate flow efficiency
efficiency_g = [b_flow_efficiency(g_t) for g_t in Gt]
efficiency_s = [b_flow_efficiency(g_s) for g_s in St]
from scipy.stats import ks_2samp
statistic, p_value = ks_2samp(efficiency_g, efficiency_s)
print(f"Flow Efficiency of the graph: {statistic:.4f}")

Flow Efficiency of the graph: 0.6000
